# 01 — Data Exploration (US1)

Run ingest → geocode → augment; summary statistics, facility counts by type/state, population totals, RUCA distribution; map facility locations; verify 8 zero-burn-center states.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
from pipeline.ingest import ingest_nird
from pipeline import config

# Use sample if full NIRD not present
nird_path = config.NIRD_FULL_PATH if config.NIRD_FULL_PATH.exists() else config.NIRD_SAMPLE_PATH
facilities, report = ingest_nird(nird_path)
print("Validation report:", report)
print("Facility counts — definitive:", report["facility_counts"]["definitive"], "stabilization:", report["facility_counts"]["stabilization"])

Ingest:   0%|          | 0/4 [00:00<?, ?step/s]

Validation report: {'rows': 635, 'duplicates_aha_id': 0, 'null_aha_id': 0, 'errors': [], 'null_counts': {'AHA_ID': 0, 'HOSPITAL_NAME': 0, 'STATE': 0, 'ZIP_CODE': 0, 'ADDRESS': 0, 'CITY': 0}, 'duplicate_aha_ids': [], 'validation_passed': True, 'facility_counts': {'definitive': 136, 'stabilization': 460, 'total': 635}}
Facility counts — definitive: 136 stabilization: 460


In [2]:
# Generate CSV for manual Census batch upload (if API fails)
import importlib
import pipeline.geocode
importlib.reload(pipeline.geocode)
from pipeline.geocode import write_batch_upload_file
from pipeline import config

# batch_path = write_batch_upload_file(facilities)
# print(f"Saved {len(facilities)} rows to:\n  {batch_path.resolve()}")
# print("\nUpload at: https://geocoding.geo.census.gov/geocoder/geographies/addressbatch")
# print("  Benchmark: Public_AR_Current, Vintage: Current_Current → Submit → download result CSV")

In [3]:
# Load manual Census batch results (GeocodeResults.csv) and merge into facilities
import importlib
import pipeline.geocode
importlib.reload(pipeline.geocode)
from pipeline.geocode import load_batch_results_and_merge, assign_tract_from_tiger, print_missing_tract_facilities
from pipeline import config

facilities = load_batch_results_and_merge(facilities, results_path=config.TABLES_DIR / "GeocodeResults.csv")
# Assign tract from TIGER for any with coords but no tract_geoid
facilities = assign_tract_from_tiger(facilities)
with_tract = (facilities["tract_geoid"].notna() & (facilities["tract_geoid"] != "")).sum()
print(f"Geocode match rate: {with_tract}/{len(facilities)} = {100*with_tract/len(facilities):.1f}%")
print_missing_tract_facilities(facilities)

  TIGER: loading 51 tract shapefiles...


TIGER load:   0%|          | 0/51 [00:00<?, ?shp/s]

  TIGER: assigned tract for 78 facilities (point-in-polygon)
Geocode match rate: 635/635 = 100.0%


In [4]:
# Augment: tract-level table (TIGER + ACS + RUCA). Requires external data downloaded.
import importlib
import pipeline.augment
importlib.reload(pipeline.augment)
from pipeline.augment import build_analytic_table
try:
    tracts = build_analytic_table(facilities)
    print("Tract count:", len(tracts))
    if "total_pop" in tracts.columns:
        print("Total population:", tracts["total_pop"].sum())
    if "is_rural" in tracts.columns:
        print("Rural tract share:", tracts["is_rural"].mean() * 100, "%")
except Exception as e:
    print("Augment skipped (missing TIGER/ACS/RUCA):", e)

Augment:   0%|          | 0/5 [00:00<?, ?step/s]

Augment TIGER:   0%|          | 0/51 [00:00<?, ?shp/s]

Tract count: 84415
Total population: 331097593
Rural tract share: 19.433749925961024 %


In [5]:
# Summary: facility counts by state and type
by_state = facilities.groupby("STATE").agg(
    total=("AHA_ID", "count"),
    definitive=("is_definitive", "sum"),
    stabilization=("is_stabilization", "sum"),
).astype(int)
print(by_state.sort_values("definitive", ascending=False).head(15))
zero_burn = by_state[by_state["definitive"] == 0].index.tolist()
print("States with zero burn centers:", len(zero_burn), "—", zero_burn[:20])

       total  definitive  stabilization
STATE                                  
CA        62          13             44
TX        48          10             32
NY        36          10             23
OH        26           8             16
MO        20           7             12
PA        36           7             27
MI        36           6             30
FL        36           6             27
IL        57           5             51
CO        19           4             14
IN         9           4              5
LA         9           4              5
MA        11           3              7
AL         6           3              3
GA        16           3             11
States with zero burn centers: 7 — ['AK', 'DE', 'MS', 'MT', 'ND', 'NH', 'SD']


In [6]:
# Map facility locations (folium)
# Red = definitive burn centers; Blue = other NIRD facilities (stabilization/trauma-only).
import pandas as pd
mapped = facilities[facilities["latitude"].notna() & facilities["longitude"].notna()]
if len(mapped) > 0:
    import folium
    m = folium.Map(location=[mapped["latitude"].mean(), mapped["longitude"].mean()], zoom_start=4)
    for _, row in mapped.iterrows():
        folium.CircleMarker([row["latitude"], row["longitude"]], radius=4, color="red" if row["is_definitive"] else "blue").add_to(m)
    display(m)
else:
    print("No coordinates to map; run geocode first.")

**To see the map:** Use **File → Trust Notebook** (or run the command *Notebook: Trust* from the Command Palette). The map will then render in the cell output.